# A10 · Gemma 4 (sliding/global · K=V · PLE · KV 공유 · MoE) 검증

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_llm_dit/blob/main/notebooks/verify/A10_gemma4.ipynb)

원본: transformers `gemma4` 의 `Gemma4TextModel` 과 `gemma4_unified` 의 `Gemma4UnifiedTextModel`
(tag `v5.20.0` = commit `48e4bcd7`; 2026-10 `main`(`683b7219`) 의 두 폴더 파일과 글자 단위로 같음).
내 구현: `miniarch/models/gemma4.py`. 텍스트 backbone 만 비교합니다 (vision/audio 인코더 제외).

체크포인트 (config.json 확인): 31B (dense, K=V), 26B-A4B (K=V + dense MLP·MoE 병렬, expert 128 중 8),
E4B / E2B (PLE + 마지막 18/20 층 KV 공유, E2B 는 공유 층 MLP 2배 폭), 12B (`gemma4_unified`, 2026-05).

부품
- sliding 5 : global 1 하이브리드. sliding 층 head_dim 256 / global 층 512, global 층 KV head 수 따로 (`num_global_key_value_heads`)
- QK-norm + V-norm(가중치 없음), softmax 스케일 1.0
- `attention_k_eq_v`: global 층은 v_proj 없이 k_proj 출력을 V 로 사용 (K 는 k_norm+RoPE, V 는 V-norm)
- RoPE 층 종류별: sliding 은 기본(θ=1e4), global 은 proportional(θ=1e6, 주파수 앞 25% 만 회전)
- Gemma sandwich norm (attention·FFN 앞뒤), 임베딩 × √d, 층 끝 `layer_scalar`, GeGLU(gelu_tanh) MLP, logit softcapping(30)
- MoE (26B-A4B): dense MLP 와 expert 가지를 둘 다 계산해 각각 norm 후 더함. router 는 norm·scale·per_expert_scale
- PLE (E 계열): 층마다 토큰별 작은 벡터를 게이트에 곱해 residual 에 주입
- KV 공유 (E 계열): 마지막 층들은 공유 직전 같은 종류 층의 K/V 를 재사용, 2배 폭 MLP 스위치

`gemma4_unified` 텍스트 모델은 PLE·MoE 항목이 빠졌을 뿐 attention·MLP·RoPE·층 계산이 gemma4 와 같은 코드입니다
(생성된 modeling 파일 diff 로 확인). 별도 스위치 없이 같은 구현으로 자체 점검에서 한 번 더 비교합니다.

축소 config (본 비교): 7층 = [S, F, S, S, F, S, F], window 5, 시퀀스 24 — 원본이 허용하는 기능을 한꺼번에 켬
- head_dim sliding 16 / global 32, KV head sliding 2 / global 1, K=V 켬 → 설정이 뒤바뀌면 shape 부터 어긋남
- global RoPE: 32 × 0.25 / 2 = 주파수 4개만 회전
- 마지막 3층 KV 공유 (F, S, F 가 각각 layer 1, 3 의 K/V 사용) + 공유 층 2배 폭 MLP
- MoE expert 8개 중 3개 + dense MLP, PLE dim 8
- 원본 초기값이 1 인 `layer_scalar`, `per_expert_scale` 과 작은 router 가중치는 0/1 이 아닌 값으로 채움

자체 점검: E 계열(K=V 끔), dense 31B 형, `Gemma4UnifiedTextModel`, `Gemma4ForCausalLM`(softcapping 된 logits) 각각 원본과 비교.

순서: 0. 환경 준비 → 1. 원본 구조 (HF 헤더) → 2. 원본 모델 동작 확인 → 3. 내 구현 == 원본 (층별 출력 비교) → 4. causal 누수 → 5. 단계 전용 자체 점검 → 6. 결과 저장

**런타임 → 런타임 유형 변경 → T4 GPU** 선택 후 위에서부터 실행하세요. 다른 모델 노트북과 독립적으로 돌아갑니다.

## 0. 환경 준비

In [ ]:
import os
REPO_DIR = "/content/tmp_llm_dit"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/HisameOgasahara/tmp_llm_dit.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip -q install -U -r requirements-colab.txt
# Colab 기본 torchao 가 최신 diffusers 와 버전이 안 맞아 import 가 깨짐 → 쓰지 않으므로 제거
!pip -q uninstall -y torchao

In [ ]:
import sys, torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (런타임 유형을 GPU로 바꾸세요)")
!git log -1 --format='repo commit: %h %s'

In [ ]:
!pip -q install -U 'transformers>=5.20.0'

In [ ]:
from miniarch import verify as V
from miniarch.verify import stages
S = stages.get("A10")
dev = "cuda" if torch.cuda.is_available() else "cpu"
try:   # gated 모델 헤더/원격 코드용. Colab 왼쪽 🔑(보안 비밀)에 HF_TOKEN 등록.
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

CHECKS = []   # (이름, 통과, 오차) → 마지막에 저장
def report(name, ok, err=float("nan")):
    CHECKS.append({"check": name, "ok": bool(ok), "max_abs": float(err)})
    print(f"{'✅' if ok else '❌'} {name:<44} max_abs={err:.2e}")

## 원본 구조 (HF 헤더)

실제 체크포인트의 safetensors 헤더만 읽어 가중치 이름, shape, dtype을 봅니다 (다운로드 없음).
**구현 검증이 아니라 원본 구조 확인용**입니다. routed expert는 개수가 많아 숨깁니다.

In [ ]:
def peek(repo, pattern, subfolder="", limit=60):
    print(f"\n===== {repo}  /{pattern}/")
    try:
        cfg = V.fetch_config(repo, subfolder)
        cfg = cfg.get("text_config", cfg)
        keys = ["model_type", "hidden_size", "num_hidden_layers", "num_attention_heads", "num_key_value_heads", "head_dim"]
        print({k: cfg[k] for k in keys if k in cfg})
        h = V.read_model_header(repo, subfolder=subfolder, pattern=pattern)
        V.show({k: v for k, v in h.items() if ".experts." not in k}, limit)
    except Exception as e:
        print(f"⚠️ {type(e).__name__}: {e}")

peek('google/gemma-4-26B-A4B-it', 'language_model\\.layers\\.(4|5)\\.', subfolder='')
peek('google/gemma-4-E4B-it', 'language_model\\.(layers\\.(5|41)\\.|per_layer|embed_tokens_per_layer)', subfolder='')

## 원본 모델 동작 확인

축소 config로 원본을 만들고 한 번 forward 합니다. 여기서 실패하면 라이브러리 버전 문제입니다 (내 구현과 무관).

In [ ]:
V.set_deterministic(0)
ref, inputs = S.ref()
ref = ref.to(dev).eval()
with torch.no_grad():
    out = ref(**{k: v.to(dev) if torch.is_tensor(v) else v for k, v in inputs.items()})
t = out[0] if isinstance(out, tuple) else getattr(out, "last_hidden_state", None)
t = t if t is not None else out.sample
print(f"{type(ref).__name__}  params={sum(p.numel() for p in ref.parameters()):,}  out={tuple(t.shape)}")

## 내 구현 == 원본

원본 가중치에 잡음을 넣고(초기값 0/1 때문에 복사 누락이 가려지지 않도록) 내 구현에 복사한 뒤, 층별 출력과 최종 출력을 비교합니다.

In [ ]:
row = V.cases.run_case(S.CASE)
print(row["status"]); print(row["detail"])
report("원본 동치 (" + S.CASE.reference + ")", row["status"].startswith("✅"))

## causal 누수

뒤쪽 토큰을 바꿨을 때 앞쪽 출력이 변하면 미래 정보가 새고 있다는 뜻입니다.

In [ ]:
import importlib
V.set_deterministic(0)
mine = importlib.import_module(S.CASE.mine_module).verify_pair(S.ref()[0])["mine"].to(dev).eval()
r = V.causal_leak_test(lambda x: mine(x).last_hidden_state, vocab_size=V.cases.VOCAB, seq_len=48, device=dev, atol=1e-4)
report("내 구현 " + r.name, r.ok, r.max_abs)

## 단계 전용 자체 점검

### 다른 변형 config 도 원본과 같은지

- `e_series`: K=V 끔(global 층에도 v_proj, KV head = sliding 과 같음) + PLE + KV 공유 + 2배 폭 MLP (E4B/E2B 형)
- `dense`: 31B 형 (K=V, PLE·공유·MoE 없음, 기본 5:1 층 배치)
- `unified`: `Gemma4UnifiedTextModel` (12B, 텍스트 경로 동일한지)
- `causal_lm`: `Gemma4ForCausalLM` 으로 감싸 logits(softcapping 포함)까지 비교

In [ ]:
from miniarch.models import gemma4 as G

def eq_variant(v):
    V.set_deterministic(0)
    r, inp = S.ref(v)
    V.perturb_params(r)
    pair = G.verify_pair(r)
    mine = pair.pop("mine")
    rep = V.check_equivalence(r, mine, inp, atol=S.CASE.atol, device=dev, **pair)
    print(f"--- {v}: {type(r).__name__}")
    print(rep.summary())
    report(f"{v} == 원본", rep.ok, max(x.max_abs for x in rep.results))

for v in ["e_series", "dense", "unified", "causal_lm"]:
    eq_variant(v)

### attention map · window · KV 공유 · proportional RoPE

- `attn_probs` 행 합 = 1, 미래 칸과 sliding window 밖 칸 확률은 정확히 0.
- KV 공유 층의 K(`k_probe`)는 넘겨준 층의 K 와 비트 단위로 같아야 합니다.
- global 층 inv_freq 중 0 이 아닌(회전하는) 주파수는 head_dim × 0.25 / 2 개.

In [ ]:
from miniarch.models import gemma4 as G
ref, inp = S.ref()
m = G.verify_pair(ref)["mine"].to(dev).eval()
c = m.config
names = [n for n, _ in m.named_modules() if n.endswith(("attn_probs", "k_probe"))]
x = inp["input_ids"].to(dev)
T_ = x.shape[1]
with V.capture(m, names) as st, torch.no_grad():
    m(x)
i_, j_ = torch.arange(T_, device=dev)[:, None], torch.arange(T_, device=dev)[None, :]
for i in range(c.n_layer):
    t, att = c.layer_types[i], m.layers[i].self_attn
    p = st[f"layers.{i}.self_attn.attn_probs"]
    outside = j_ > i_
    if t == "sliding_attention":
        outside = outside | (i_ - j_ >= c.sliding_window)
    err = (p.sum(-1) - 1).abs().max().item()
    leak = p[..., outside].abs().max().item()
    print(f"layer {i} {t:<18} head_dim={att.d:<3} kv={att.kv} 공유={att.is_shared!s:<5} 넘겨줌={att.store_kv!s:<5} "
          f"행합 오차={err:.1e} 마스크 밖={leak:.1e}")
    report(f"layer {i} 확률 합", err < 1e-5, err)
    report(f"layer {i} 마스크 밖 확률 0", leak == 0, leak)
src = {c.layer_types[i]: i for i in range(c.n_layer) if m.layers[i].self_attn.store_kv}
for i in range(c.n_layer):
    if m.layers[i].self_attn.is_shared:
        s = src[c.layer_types[i]]
        d = (st[f"layers.{i}.self_attn.k_probe"] - st[f"layers.{s}.self_attn.k_probe"]).abs().max().item()
        report(f"layer {i} K == layer {s} K (KV 공유)", d == 0, d)
inv = m.full_attention_inv_freq
n_rot = int((inv > 0).sum())
print(f"global inv_freq: 회전 {n_rot} / {inv.numel()}")
report("global RoPE 앞 25% 만 회전", n_rot == int(0.25 * c.global_head_dim // 2), 0.0)

### ablation 스위치

스위치를 끄면 출력이 실제로 바뀌는지 확인합니다 (같은 가중치를 그대로 로드). softcapping 은 logits 가 ±cap 안에 있는지만 봅니다.

In [ ]:
import dataclasses
from miniarch.models import gemma4 as G
ref, inp = S.ref()
m = G.verify_pair(ref)["mine"].to(dev).eval()
x = inp["input_ids"].to(dev)
with torch.no_grad():
    out = m(x)
for sw in ["qk_norm", "v_norm", "sandwich_norm", "scale_embed", "use_dense_branch", "use_moe_branch", "use_ple"]:
    m2 = G.Gemma4(dataclasses.replace(m.config, **{sw: False})).to(dev).eval()
    m2.load_state_dict(m.state_dict())
    with torch.no_grad():
        d = (m2(x).last_hidden_state - out.last_hidden_state).abs().max().item()
    report(f"{sw}=False 이면 출력이 바뀜", d > 1e-4, d)
cap = m.config.final_logit_softcap
mx = out.logits.abs().max().item()
report(f"logits 가 ±{cap} 안", mx < cap, mx)

## 결과 저장

- `results/`에 점검 결과를 저장합니다.
- 노트북 출력까지 남기려면 **파일 → GitHub에 사본 저장** → 이 저장소의 `notebooks/verify/A10_gemma4.ipynb`.

In [ ]:
import datetime, pandas as pd
os.makedirs("results", exist_ok=True)
df = pd.DataFrame(CHECKS)
p = f"results/verify_A10_{datetime.datetime.now():%Y%m%d-%H%M}.csv"
df.to_csv(p, index=False)
print(df.to_string(index=False)); print("saved:", p)
print("전체:", "✅ 모두 통과" if df.ok.all() else "❌ 실패 있음")